# Credit Scoring for the Unbanked
## Home Credit Default Risk — Model Training & Comparison (Improved)

**Models**: Logistic Regression · Random Forest · Gradient Boosting · XGBoost · LightGBM · CatBoost
**Imbalance Mitigation**: `scale_pos_weight` / `class_weight='balanced'` / `sample_weight`
**Target**: Predict loan default (TARGET = 1)

Perbaikan dari versi sebelumnya:
1. Tambah fitur EXT_SOURCE_MEAN, EXT_SOURCE_MIN, EXT_SOURCE_PROD
2. Fix Gradient Boosting menggunakan sample_weight agar Recall tidak collapse ke 0
3. Threshold optimal via Precision-Recall curve untuk semua model
4. Tambah Gini Coefficient di evaluasi
5. Simpan model terbaik ke file pkl

In [9]:
%pip install xgboost lightgbm catboost imbalanced-learn scikit-learn pandas numpy matplotlib seaborn joblib -q

Note: you may need to restart the kernel to use updated packages.


In [10]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, classification_report,
    precision_recall_curve
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

print("Semua library berhasil diimport.")

Semua library berhasil diimport.


## 1. Load Data & Feature Engineering

In [8]:
# Load data
df = pd.read_csv("home-credit-default-risk/application_train.csv")
print(df.shape)

# Fix anomali DAYS_EMPLOYED (365243 = kode dummy tidak bekerja)
df["DAYS_EMPLOYED_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
df["DAYS_EMPLOYED"]      = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# Fitur dasar
df["AGE_YEARS"]      = abs(df["DAYS_BIRTH"]) / 365
df["DEBT_TO_INCOME"] = df["AMT_CREDIT"]  / (df["AMT_INCOME_TOTAL"] + 1)
df["PAYMENT_RATE"]   = df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + 1)

# [FIX 1] Gabungkan EXT_SOURCE — fitur paling prediktif di dataset ini
ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
df["EXT_SOURCE_MIN"]  = df[ext_cols].min(axis=1)
df["EXT_SOURCE_PROD"] = df[ext_cols].prod(axis=1)

print("Feature engineering selesai.")

(307511, 122)
Feature engineering selesai.


## 2. Select Features & Split Data

In [11]:
features = [
    # Demografi
    "CODE_GENDER", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "OCCUPATION_TYPE",
    "FLAG_OWN_CAR", "FLAG_OWN_REALTY",

    # Keuangan
    "CNT_CHILDREN", "CNT_FAM_MEMBERS",
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",

    # Waktu & pekerjaan
    "DAYS_EMPLOYED", "DAYS_LAST_PHONE_CHANGE",
    "AGE_YEARS", "DAYS_EMPLOYED_ANOM",

    # Skor eksternal
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "EXT_SOURCE_MEAN", "EXT_SOURCE_MIN", "EXT_SOURCE_PROD",

    # Fitur engineering
    "DEBT_TO_INCOME", "PAYMENT_RATE",
]

X = df[features]
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train : {X_train.shape}")
print(f"Test  : {X_test.shape}")
print(f"Fitur : {X_train.shape[1]}")

Train : (246008, 25)
Test  : (61503, 25)
Fitur : 25


## 3. Preprocessing Pipeline

In [12]:
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols     = X_train.select_dtypes(exclude=["object"]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# Encode data untuk model yang tidak pakai Pipeline
X_train_enc = preprocessor.fit_transform(X_train)
X_test_enc  = preprocessor.transform(X_test)

ratio = (y_train == 0).sum() / (y_train == 1).sum()
sw    = compute_sample_weight("balanced", y_train)

print(f"scale_pos_weight : {ratio:.2f}")
print("Preprocessing selesai.")

scale_pos_weight : 11.39
Preprocessing selesai.


## 4. Fungsi Evaluasi

In [13]:
def find_optimal_threshold(y_true, y_prob):
    prec, rec, thresh = precision_recall_curve(y_true, y_prob)
    f1    = np.where((prec + rec) == 0, 0, 2 * prec * rec / (prec + rec))
    best  = np.argmax(f1[:-1])
    return thresh[best]

def evaluate_model(name, y_true, y_prob, threshold=None):
    if threshold is None:
        threshold = find_optimal_threshold(y_true, y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    auc    = roc_auc_score(y_true, y_prob)

    print(f"===== {name} =====")
    print(f"AUC-ROC   : {auc:.4f}")
    print(f"Gini      : {2*auc-1:.4f}")
    print(f"Threshold : {threshold:.3f}")
    print(f"Accuracy  : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision : {precision_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"Recall    : {recall_score(y_true, y_pred, zero_division=0):.4f}")
    print(f"F1 Score  : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    print(classification_report(y_true, y_pred, zero_division=0))
    return auc, threshold

results = {}
print("Fungsi evaluasi siap.")

Fungsi evaluasi siap.


## 5. Training Model

### Model 1: Logistic Regression

In [14]:
lr_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42))
])
lr_pipeline.fit(X_train, y_train)

lr_prob = lr_pipeline.predict_proba(X_test)[:, 1]
auc, thr = evaluate_model("Logistic Regression", y_test, lr_prob)
results["Logistic Regression"] = {"auc": auc, "threshold": thr, "prob": lr_prob}

===== Logistic Regression =====
AUC-ROC   : 0.7453
Gini      : 0.4907
Threshold : 0.651
Accuracy  : 0.8357
Precision : 0.2257
Recall    : 0.4260
F1 Score  : 0.2951
              precision    recall  f1-score   support

           0       0.95      0.87      0.91     56538
           1       0.23      0.43      0.30      4965

    accuracy                           0.84     61503
   macro avg       0.59      0.65      0.60     61503
weighted avg       0.89      0.84      0.86     61503



C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18540\1079318936.py:3: RuntimeWarning: invalid value encountered in divide
  f1    = np.where((prec + rec) == 0, 0, 2 * prec * rec / (prec + rec))


### Model 2: Random Forest

In [15]:
rf_pipeline = Pipeline([
    ("prep",  preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300, max_depth=10,
        class_weight="balanced", random_state=42, n_jobs=-1))
])
rf_pipeline.fit(X_train, y_train)

rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]
auc, thr = evaluate_model("Random Forest", y_test, rf_prob)
results["Random Forest"] = {"auc": auc, "threshold": thr, "prob": rf_prob}

===== Random Forest =====
AUC-ROC   : 0.7478
Gini      : 0.4956
Threshold : 0.620
Accuracy  : 0.8448
Precision : 0.2352
Recall    : 0.4099
F1 Score  : 0.2989
              precision    recall  f1-score   support

           0       0.94      0.88      0.91     56538
           1       0.24      0.41      0.30      4965

    accuracy                           0.84     61503
   macro avg       0.59      0.65      0.61     61503
weighted avg       0.89      0.84      0.86     61503



C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18540\1079318936.py:3: RuntimeWarning: invalid value encountered in divide
  f1    = np.where((prec + rec) == 0, 0, 2 * prec * rec / (prec + rec))


### Model 3: Gradient Boosting

Perbaikan: pakai `sample_weight` saat `.fit()` karena Gradient Boosting
tidak support `class_weight` langsung. Versi sebelumnya Recall = 0.01.

In [16]:
gb_model = GradientBoostingClassifier(
    n_estimators=150, learning_rate=0.05, random_state=42)

# [FIX 2] Pakai sample_weight bukan class_weight
gb_model.fit(X_train_enc, y_train, sample_weight=sw)

gb_prob = gb_model.predict_proba(X_test_enc)[:, 1]
auc, thr = evaluate_model("Gradient Boosting", y_test, gb_prob)
results["Gradient Boosting"] = {"auc": auc, "threshold": thr, "prob": gb_prob}

===== Gradient Boosting =====
AUC-ROC   : 0.7553
Gini      : 0.5106
Threshold : 0.650
Accuracy  : 0.8431
Precision : 0.2403
Recall    : 0.4365
F1 Score  : 0.3099
              precision    recall  f1-score   support

           0       0.95      0.88      0.91     56538
           1       0.24      0.44      0.31      4965

    accuracy                           0.84     61503
   macro avg       0.59      0.66      0.61     61503
weighted avg       0.89      0.84      0.86     61503



C:\Users\LENOVO\AppData\Local\Temp\ipykernel_18540\1079318936.py:3: RuntimeWarning: invalid value encountered in divide
  f1    = np.where((prec + rec) == 0, 0, 2 * prec * rec / (prec + rec))


### Model 4: XGBoost

In [17]:
xgb = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=ratio, eval_metric="logloss",
    random_state=42, verbosity=0)
xgb.fit(X_train_enc, y_train)

xgb_prob = xgb.predict_proba(X_test_enc)[:, 1]
auc, thr = evaluate_model("XGBoost", y_test, xgb_prob)
results["XGBoost"] = {"auc": auc, "threshold": thr, "prob": xgb_prob}

===== XGBoost =====
AUC-ROC   : 0.7640
Gini      : 0.5281
Threshold : 0.687
Accuracy  : 0.8665
Precision : 0.2699
Recall    : 0.3837
F1 Score  : 0.3169
              precision    recall  f1-score   support

           0       0.94      0.91      0.93     56538
           1       0.27      0.38      0.32      4965

    accuracy                           0.87     61503
   macro avg       0.61      0.65      0.62     61503
weighted avg       0.89      0.87      0.88     61503



### Model 5: LightGBM

In [18]:
lgbm = LGBMClassifier(
    n_estimators=300, learning_rate=0.05,
    num_leaves=31, class_weight="balanced",
    random_state=42, verbose=-1)
lgbm.fit(X_train_enc, y_train)

lgbm_prob = lgbm.predict_proba(X_test_enc)[:, 1]
auc, thr = evaluate_model("LightGBM", y_test, lgbm_prob)
results["LightGBM"] = {"auc": auc, "threshold": thr, "prob": lgbm_prob}

c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


===== LightGBM =====
AUC-ROC   : 0.7641
Gini      : 0.5281
Threshold : 0.680
Accuracy  : 0.8583
Precision : 0.2588
Recall    : 0.4054
F1 Score  : 0.3159
              precision    recall  f1-score   support

           0       0.95      0.90      0.92     56538
           1       0.26      0.41      0.32      4965

    accuracy                           0.86     61503
   macro avg       0.60      0.65      0.62     61503
weighted avg       0.89      0.86      0.87     61503



### Model 6: CatBoost

In [19]:
cat_features_idx = [X_train.columns.get_loc(c) for c in categorical_cols]

X_train_cat = X_train.copy()
X_test_cat  = X_test.copy()
for col in categorical_cols:
    X_train_cat[col] = X_train_cat[col].fillna("Missing")
    X_test_cat[col]  = X_test_cat[col].fillna("Missing")
for col in numeric_cols:
    med = X_train_cat[col].median()
    X_train_cat[col] = X_train_cat[col].fillna(med)
    X_test_cat[col]  = X_test_cat[col].fillna(med)

cat = CatBoostClassifier(
    iterations=300, learning_rate=0.05, depth=6,
    class_weights=[1, ratio], verbose=0, random_state=42)
cat.fit(X_train_cat, y_train, cat_features=cat_features_idx)

cat_prob = cat.predict_proba(X_test_cat)[:, 1]
auc, thr = evaluate_model("CatBoost", y_test, cat_prob)
results["CatBoost"] = {"auc": auc, "threshold": thr, "prob": cat_prob}

===== CatBoost =====
AUC-ROC   : 0.7628
Gini      : 0.5257
Threshold : 0.691
Accuracy  : 0.8671
Precision : 0.2697
Recall    : 0.3780
F1 Score  : 0.3148
              precision    recall  f1-score   support

           0       0.94      0.91      0.93     56538
           1       0.27      0.38      0.31      4965

    accuracy                           0.87     61503
   macro avg       0.61      0.64      0.62     61503
weighted avg       0.89      0.87      0.88     61503



## 6. Perbandingan Semua Model

In [22]:
import pandas as pd

summary = []
for name, r in results.items():
    prob  = r["prob"]
    thr   = r["threshold"]
    pred  = (prob >= thr).astype(int)
    auc   = r["auc"]
    summary.append({
        "Model"    : name,
        "AUC-ROC"  : round(auc, 4),
        "Gini"     : round(2*auc-1, 4),
        "F1"       : round(f1_score(y_test, pred, zero_division=0), 4),
        "Precision": round(precision_score(y_test, pred, zero_division=0), 4),
        "Recall"   : round(recall_score(y_test, pred, zero_division=0), 4),
        "Threshold": round(thr, 3),
    })

df_summary = pd.DataFrame(summary).sort_values("AUC-ROC", ascending=False).reset_index(drop=True)
df_summary.index += 1
print(df_summary.to_string())
print()
print(f"Model terbaik : {df_summary.iloc[0]['Model']}")
print(f"AUC terbaik   : {df_summary.iloc[0]['AUC-ROC']}")
print(f"Gini terbaik  : {df_summary.iloc[0]['Gini']}")

                 Model  AUC-ROC    Gini      F1  Precision  Recall  Threshold
1             LightGBM   0.7641  0.5281  0.3159     0.2588  0.4054      0.680
2              XGBoost   0.7640  0.5281  0.3169     0.2699  0.3837      0.687
3             CatBoost   0.7628  0.5257  0.3148     0.2697  0.3780      0.691
4    Gradient Boosting   0.7553  0.5106  0.3099     0.2403  0.4365      0.650
5        Random Forest   0.7478  0.4956  0.2989     0.2352  0.4099      0.620
6  Logistic Regression   0.7453  0.4907  0.2951     0.2257  0.4260      0.651

Model terbaik : LightGBM
AUC terbaik   : 0.7641
Gini terbaik  : 0.5281


## 7. Simpan Model Terbaik

In [21]:
best_name = df_summary.iloc[0]["Model"]
best_thr  = results[best_name]["threshold"]

model_map = {
    "Logistic Regression": lr_pipeline,
    "Random Forest"      : rf_pipeline,
    "Gradient Boosting"  : gb_model,
    "XGBoost"            : xgb,
    "LightGBM"           : lgbm,
    "CatBoost"           : cat,
}

best_model = model_map[best_name]

joblib.dump({
    "model"       : best_model,
    "threshold"   : best_thr,
    "model_name"  : best_name,
    "preprocessor": preprocessor,
    "features"    : features,
}, "best_model_abel.pkl")

print(f"Model tersimpan : best_model_abel.pkl")
print(f"Model terbaik   : {best_name}")
print(f"Threshold opt.  : {best_thr:.4f}")
print()
print("Cara load dan prediksi:")
print("  import joblib")
print("  obj   = joblib.load('best_model_abel.pkl')")
print("  model = obj['model']")
print("  thr   = obj['threshold']")
print("  prob  = model.predict_proba(X_baru)[:, 1]")
print("  pred  = (prob >= thr).astype(int)")

Model tersimpan : best_model_abel.pkl
Model terbaik   : LightGBM
Threshold opt.  : 0.6799

Cara load dan prediksi:
  import joblib
  obj   = joblib.load('best_model_abel.pkl')
  model = obj['model']
  thr   = obj['threshold']
  prob  = model.predict_proba(X_baru)[:, 1]
  pred  = (prob >= thr).astype(int)
